[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/Datacube-demo/blob/main/GEE/GEE.ipynb)

## Google Earth Engine
Google Earth Engine combines a multi-petabyte catalog of satellite imagery and geospatial datasets with planetary-scale analysis capabilities. Scientists, researchers, and developers use Earth Engine to detect changes, map trends, and quantify differences on the Earth's surface.   

![gee_overview.png](https://raw.githubusercontent.com/NTU-CompHydroMet-Lab/Datacube-demo/main/GEE/images/gee_overview.png)  
Source: [geohackweek](https://geohackweek.github.io/GoogleEarthEngine/01-introduction/)

### 中文說明

Google Earth Engine（GEE）可以理解為一個 **Google 代管的超大型資料立方**：
超過數 PB 的衛星影像與地理資料已經整理好放在雲端，並提供行星尺度的運算能力。
與自建的 Open Data Cube 不同，你**不需要建置任何環境、不需要下載資料**——
分析以「查詢」的形式送到 Google 的伺服器執行，只有最終結果回到你手上。

本教學示範最常見的工作流程：從資料目錄取一個影像集合（ImageCollection）、
做時空篩選與合成，最後畫在互動地圖上。

### What can you get from Google Earth Engine?
Nighttime light imagery after sunset from NOAA   
![noaa_night_lights.png](https://raw.githubusercontent.com/NTU-CompHydroMet-Lab/Datacube-demo/main/GEE/images/noaa_night_lights.png)   
Source: [NOAA](https://www.ncei.noaa.gov/news/sunset-nighttime-lights-noaa)

GEE 的公開資料目錄涵蓋光學／雷達衛星影像、地形、土地覆蓋、氣象等超過千種資料集，
下圖為 NOAA 的全球夜間燈光影像範例。完整目錄見
[Earth Engine Data Catalog](https://developers.google.com/earth-engine/datasets)。

### 匯入套件

在 Colab 上 `earthengine-api` 與 `geemap` 皆已預裝，直接匯入即可；
本地環境則先執行下方註解中的 pip 安裝指令。
`ee` 是 GEE 的官方 Python API，`geemap` 提供互動式地圖顯示。

In [3]:
# Recommended installation:
#     pip install earthengine-api geemap xarray pandas matplotlib

# For data access and handling
import ee # earthengine-api
import xarray

# For plotting
import matplotlib.pyplot as plt
import pandas as pd
import geemap

### Initialize Google Earth Engine

[Google Earth Engine (GEE)](https://earthengine.google.com/) is a cloud-based platform
for planetary-scale geospatial analysis. All computation happens on Google's servers —
you don't download data, you send queries.

Either way, you need a **Google Cloud project with the Earth Engine API enabled**:
- https://scribehow.com/viewer/1_Create_a_Google_Cloud_Project_and_Enable_Earth_Engine_API__oobJRoZcRoGYFZngARq2IQ

**Option 1 — Interactive login (recommended on Colab).** `ee.Authenticate()` opens a
Google login prompt; grant access, then initialize with your project ID.

**Option 2 — Service account (for unattended scripts / servers).** Authenticate with a
JSON key file instead of a personal login:
- https://scribehow.com/viewer/2_Generate_Google_Earth_Engine_Service_Account_and_json_Key__SJPnajZQQCq-TU5hc0nT6g

### 中文說明

使用 GEE 前需要：**一個已啟用 Earth Engine API 的 Google Cloud 專案**（上方教學連結有申請步驟）。認證方式二選一：

- **選項 1（Colab 建議）**：`ee.Authenticate()` 互動式登入——執行後跳出 Google 授權視窗，
  登入並同意後，把你的 Cloud 專案 ID 填入 `GEE_PROJECT` 即可。
- **選項 2（伺服器／排程用）**：Service Account + JSON 金鑰，適合無人值守的程式，
  請將下方註解取消並填入金鑰資訊。

In [ ]:
# Option 1: interactive login (works on Colab and local Jupyter)
GEE_PROJECT = "PLEASE ENTER YOUR GOOGLE CLOUD PROJECT ID"

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

In [ ]:
# Option 2: service account with a JSON key file (uncomment to use)
# service_account = "PLEASE ENTER YOUR SERVICE ACCOUNT EMAIL"  # client_email in json file
# key_path        = "PLEASE ENTER THE PATH TO YOUR JSON KEY FILE"
# credentials     = ee.ServiceAccountCredentials(service_account, key_path)
# ee.Initialize(credentials)

### 範例：2023 年台灣夜間燈光合成圖

以下用 NOAA VIIRS 月合成夜間燈光資料，示範 GEE 的典型分析流程：

1. **定義範圍**（ROI）：台灣的經緯度範圍框；
2. **取得影像集合**：`ee.ImageCollection` 指向雲端目錄中的資料集；
3. **時空篩選與合成**：`filterBounds` 空間篩選 → `filterDate` 取 2023 全年
   → `mean()` 把 12 張月合成影像平均成一張（降低雲層與雜訊影響）；
4. **遮罩**：把輻射值過低（無燈光）的區域設為透明；
5. **視覺化**：`geemap` 互動地圖 + 暗色底圖 + 燈光漸層配色。

值得注意的是：這整段程式**沒有下載任何影像**——所有篩選、平均、遮罩都在
Google 伺服器上執行（延遲運算，lazy computation），地圖顯示時才把渲染結果串流回來。
這與前一份 ODC 教學「查詢取代搬檔案」的精神一致，差別在於 GEE 的資料與算力由
Google 代管，而 ODC 讓你**用同樣的概念管理自己的資料**——兩者正是「跨數據立方」
要整合比較的兩種代表性架構。

In [ ]:
# 1. 定義感興趣的區域 (ROI) - 設定為台灣的 Bounding Box
roi = ee.Geometry.Rectangle([119.3, 21.8, 122.1, 25.4])

# 2. 詢問呼叫 NOAA VIIRS 夜間燈光 ImageCollection
viirs_collection = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")

# 3. 從 Collection 萃取單張 Image
# 這裡計算平均值來消除單月的雲層或雜訊干擾
nighttime_image = (
    viirs_collection
    .filterBounds(roi)                      # 空間篩選
    .filterDate('2023-01-01', '2023-12-31') # 時間篩選  取 2023 年全年的影像
    .select('avg_rad')                      # 指定「平均輻射亮度」波段
    .mean()                                 # 將多張影像降維成單張 ee.Image (取平均)
    .clip(roi)                              # 裁切邊界
)

# 4. 遮罩處理 (Masking) - 讓無光區變透明，只保留發光區域
light_mask = nighttime_image.gt(0.1)    # 輻射值小於 0.1 的區域視為無燈光（背景），將其 mask 掉
nighttime_image = nighttime_image.updateMask(light_mask)

# 5. 設定視覺化與地圖渲染
Map = geemap.Map(center=[23.6, 120.9], zoom=7)
Map.add_basemap('CartoDB.DarkMatter')

# 設定 顏色漸層：純黑 > 暗褐色 > 琥珀橘 > 金黃色 > 白光
vis_params = {
    'min': 1.0,
    'max': 60.0,
    'palette': ['050505', '4d2600', 'b35900', 'ff9900', 'ffdb4d', 'ffffff']
}

Map.addLayer(nighttime_image, vis_params, 'VIIRS 2023 Night Lights')

# 顯示地圖
Map

## 小結

| | Google Earth Engine | Open Data Cube |
| --- | --- | --- |
| 資料 | Google 代管的公開目錄（PB 級） | 自行匯入、自主管理 |
| 運算 | Google 雲端（免建置） | 自建環境（Docker／伺服器） |
| 適合 | 快速取用全球公開資料 | 機敏／在地資料、客製產品 |
| 成本 | 免費額度＋用量計費 | 硬體與維運成本 |

兩者互補：公開的全球資料用 GEE 快速取得，在地或機敏資料則放進自建的 ODC。
ODC 的完整教學見本 repo 的
[`01_odc_load_demo.ipynb`](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/Datacube-demo/blob/main/ODC/notebooks/01_odc_load_demo.ipynb)。